In [1]:
import re
import time
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE_URL = "https://pewneauto.pl/oferty/brand/toyota,lexus/station-id/162,180/_sort/new" #@param {type:"string"}
BASE_LISTING_URL = BASE_URL + "/na-strone=24&strona={page}"

print(BASE_LISTING_URL)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; GoogleColabScraper/1.0; +https://example.com)"
}

https://pewneauto.pl/oferty/brand/toyota,lexus/station-id/162,180/_sort/new/na-strone=24&strona={page}


In [6]:
def get_soup(url, session, sleep_time=0.5):
    resp = session.get(url, headers=HEADERS, timeout=15)
    if resp.status_code == 404:
        return None
    resp.raise_for_status()
    time.sleep(sleep_time)
    return BeautifulSoup(resp.text, "lxml")

def collect_offer_links(session, max_pages=5):
    offer_urls = set()
    page = 1
    PAGINATION_URL = BASE_URL + "?strona={page}"

    while page <= max_pages:
        url = PAGINATION_URL.format(page=page)
        print(f"Przeszukuję listing: {url}")
        soup = get_soup(url, session)

        if soup is None:
            print(f"  Strona {page} nie istnieje (404). Kończę szukanie linków.")
            break

        anchors = soup.find_all("a", href=True)
        page_links = {urljoin(url, a['href']) for a in anchors if "/oferta/" in a['href']}

        if not page_links:
            break

        new_links = page_links - offer_urls
        print(f"  Strona {page}: znaleziono {len(page_links)} linków (nowych: {len(new_links)})")

        if not new_links and page > 1:
             break

        offer_urls.update(page_links)
        page += 1
    return sorted(list(offer_urls))

def extract_price_pln(soup):
    price_tag = soup.find(class_=re.compile("price|amount", re.I))
    if price_tag:
        text = price_tag.get_text().replace(" ", "").replace("\u00a0", "").replace("zł", "")
        match = re.search(r"(\d+)", text)
        return match.group(1) if match else None
    return None

def extract_make_model_trim_location(soup):
    title = soup.find("h1")
    make_model = title.get_text(strip=True) if title else "Nieznany"

    trim_tag = soup.find(class_=re.compile("subtitle|variant|version", re.I))
    trim = trim_tag.get_text(strip=True) if trim_tag else None

    loc = "Nieznana"
    loc_label = soup.find(string=re.compile("Lokalizacja|Diler", re.I))
    if loc_label and loc_label.parent:
        val = loc_label.parent.find_next(string=True)
        if val: loc = val.strip()

    return make_model, trim, loc

def extract_specs(soup):
    data = {}
    specs_labels = {"Przebieg": "mileage_km", "Rok produkcji": "year", "Moc": "power_hp", "Pojemność": "capacity_cm3"}
    for item in soup.find_all(["li", "div"], class_=re.compile("spec|parameter|attribute", re.I)):
        txt = item.get_text(" ", strip=True)
        for label, key in specs_labels.items():
            if label.lower() in txt.lower() and ":" in txt:
                data[key] = txt.split(":")[-1].strip()
    return data

def scrape_offer(session, url):
    soup = get_soup(url, session)
    if not soup: return None
    make_model, trim, location = extract_make_model_trim_location(soup)
    price = extract_price_pln(soup)
    specs = extract_specs(soup)
    return {"url": url, "make_model": make_model, "trim": trim, "price_pln": price, "location": location, **specs}

def main(pages=100, sample_data=False):
    session = requests.Session()
    links = collect_offer_links(session, max_pages=pages)
    if sample_data: links = links[:5]
    results = []
    for i, url in enumerate(links):
        print(f"[{i+1}/{len(links)}] Pobieram dane z: {url}")
        try:
            res = scrape_offer(session, url)
            if res: results.append(res)
        except Exception as e:
            print(f"  Błąd przy {url}: {e}")
            continue
    df = pd.DataFrame(results)
    if not df.empty:
        display(df.head())
        df.to_csv("toyota_offers.csv", index=False)
        print(f"Zapisano {len(df)} ofert do pliku toyota_offers.csv")
    else:
        print("Nie znaleziono żadnych ofert.")

In [7]:
# Uruchom tę komórkę ponownie. Teraz błąd 404 zostanie obsłużony poprawnie.
main(sample_data=False)

Przeszukuję listing: https://pewneauto.pl/oferty/brand/toyota,lexus/station-id/162,180/_sort/new?strona=1
  Strona 1: znaleziono 20 linków (nowych: 20)
Przeszukuję listing: https://pewneauto.pl/oferty/brand/toyota,lexus/station-id/162,180/_sort/new?strona=2
  Strona 2: znaleziono 20 linków (nowych: 20)
Przeszukuję listing: https://pewneauto.pl/oferty/brand/toyota,lexus/station-id/162,180/_sort/new?strona=3
  Strona 3: znaleziono 20 linków (nowych: 20)
Przeszukuję listing: https://pewneauto.pl/oferty/brand/toyota,lexus/station-id/162,180/_sort/new?strona=4
  Strona 4: znaleziono 20 linków (nowych: 20)
Przeszukuję listing: https://pewneauto.pl/oferty/brand/toyota,lexus/station-id/162,180/_sort/new?strona=5
  Strona 5: znaleziono 20 linków (nowych: 20)
Przeszukuję listing: https://pewneauto.pl/oferty/brand/toyota,lexus/station-id/162,180/_sort/new?strona=6
  Strona 6: znaleziono 20 linków (nowych: 20)
Przeszukuję listing: https://pewneauto.pl/oferty/brand/toyota,lexus/station-id/162,180/_

,url,make_model,trim,price_pln,location
0,https://pewneauto.pl/oferta/lexus-ct/390317,Lexus CT,None,2316,Znajdź dilera
1,https://pewneauto.pl/oferta/lexus-ux/407950,Lexus UX,None,2911,Znajdź dilera
2,https://pewneauto.pl/oferta/toyota-auris/407012,Toyota Auris,None,1387,Znajdź dilera
3,https://pewneauto.pl/oferta/toyota-avensis/398933,Toyota Avensis,None,56900,Znajdź dilera
4,https://pewneauto.pl/oferta/toyota-aygo-x/404944,Toyota Aygo X,None,906,Znajdź dilera


Zapisano 173 ofert do pliku toyota_offers.csv
